# 06. BM25 Keyword-Based Retrieval Baseline

BM25 키워드 기반 검색으로 retrieval pipeline의 baseline 성능을 측정합니다.

| Item | Detail |
|------|--------|
| Task | QA pair retrieval (질문 -> 정답 답변 검색) |
| Method | BM25Okapi (rank_bm25) |
| Tokenizer | Whitespace split + 특수문자 제거 |
| Metrics | Recall@k, MRR@k (k=1,3,5,10,20) |
| Environment | Kaggle T4 x2 (CPU-only workload) |
| Purpose | Retrieval baseline for comparison with SBERT (notebook 07) |

---
## 0. Environment Setup

In [ ]:
%%capture
!pip install -q rank_bm25 plotly kaleido

In [ ]:
import os, json, time, re, warnings
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'iframe'
warnings.filterwarnings('ignore')

# --- Kaggle vs Local path detection ---
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/civilcomplaint-processed'
    OUT_DIR = '/kaggle/working'
    IS_KAGGLE = True
else:
    DATA_DIR = '../data/processed'
    OUT_DIR = '..'
    IS_KAGGLE = False

RESULTS_DIR = os.path.join(OUT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data: {DATA_DIR}")
print(f"Results: {RESULTS_DIR}")

---
## 1. QA Data Loading

In [ ]:
qa_path = os.path.join(DATA_DIR, 'qa_pairs.parquet')
df_qa = pd.read_parquet(qa_path)

print(f"Total QA pairs: {len(df_qa):,}")
print(f"Columns: {list(df_qa.columns)}")
print(f"\n=== Domain Distribution ===")
print(df_qa['domain'].value_counts())
print(f"\n=== Category Count ===")
print(f"Unique categories: {df_qa['category'].nunique()}")
df_qa.head(3)

In [ ]:
# QA pair distribution by domain
domain_counts = df_qa['domain'].value_counts()

fig = px.bar(
    x=domain_counts.index,
    y=domain_counts.values,
    labels={'x': 'Domain', 'y': 'QA Pairs'},
    title='QA Pair Distribution by Domain',
    color=domain_counts.values,
    color_continuous_scale='Blues',
)
fig.update_traces(texttemplate='%{y:,.0f}', textposition='outside')
fig.update_layout(
    xaxis_tickangle=-30, height=450, showlegend=False,
    margin=dict(t=60, b=80),
)
fig.show(renderer='iframe')

In [ ]:
# --- Train / Test split for retrieval evaluation ---
# Corpus = all QA pairs; Test = 1000 held-out queries
np.random.seed(42)
N_TEST = min(1000, len(df_qa) // 5)

test_indices = np.random.choice(len(df_qa), size=N_TEST, replace=False)
test_mask = np.zeros(len(df_qa), dtype=bool)
test_mask[test_indices] = True

df_test = df_qa.iloc[test_indices].reset_index(drop=True)
df_corpus = df_qa.reset_index(drop=True)  # full corpus (including test, for realistic retrieval)

test_queries = df_test['question'].tolist()
test_answers = df_test['answer'].tolist()
test_domains = df_test['domain'].tolist()
corpus_answers = df_corpus['answer'].tolist()

print(f"Corpus size: {len(df_corpus):,}")
print(f"Test queries: {len(test_queries):,}")
print(f"\nTest domain distribution:")
print(df_test['domain'].value_counts())

---
## 2. Korean Text Tokenization

In [ ]:
# --- Korean tokenizer ---
# Try Mecab first; fall back to whitespace split
USE_MECAB = False
try:
    from konlpy.tag import Mecab
    _mecab = Mecab()
    _mecab.morphs('테스트 문장입니다')
    USE_MECAB = True
    print('Tokenizer: Mecab (konlpy)')
except Exception:
    print('Tokenizer: Whitespace split (Mecab not available)')

# Preprocessing regex
_special_re = re.compile(r'[^\w\s]', re.UNICODE)

def tokenize(text: str) -> list:
    """Tokenize Korean text. Returns list of tokens."""
    if not isinstance(text, str) or len(text.strip()) == 0:
        return []
    # Remove special characters, lowercase
    text = _special_re.sub(' ', text).lower().strip()
    if USE_MECAB:
        return _mecab.morphs(text)
    else:
        return text.split()

# Quick test
sample = df_corpus['question'].iloc[0] if len(df_corpus) > 0 else '테스트 문장입니다'
print(f"Original : {sample}")
print(f"Tokenized: {tokenize(sample)}")

---
## 3. BM25 Index Construction

In [ ]:
# --- Tokenize the entire corpus ---
# Use answer text as the document corpus for retrieval
print("Tokenizing corpus ...")
t0 = time.time()

corpus_texts = df_corpus['answer'].fillna('').tolist()
tokenized_corpus = [tokenize(doc) for doc in corpus_texts]

tok_time = time.time() - t0
print(f"Tokenization done: {tok_time:.1f}s")

# Document length stats
doc_lengths = [len(tokens) for tokens in tokenized_corpus]
print(f"\n=== Corpus Stats ===")
print(f"  Documents : {len(tokenized_corpus):,}")
print(f"  Avg length: {np.mean(doc_lengths):.1f} tokens")
print(f"  Med length: {np.median(doc_lengths):.0f} tokens")
print(f"  Min length: {np.min(doc_lengths)}")
print(f"  Max length: {np.max(doc_lengths)}")

In [ ]:
# --- Build BM25 index ---
print("Building BM25Okapi index ...")
t0 = time.time()
bm25 = BM25Okapi(tokenized_corpus)
build_time = time.time() - t0

print(f"Index built: {build_time:.1f}s")
print(f"Corpus size : {bm25.corpus_size:,}")
print(f"Avg doc len : {bm25.avgdl:.1f}")

---
## 4. Retrieval Evaluation

In [ ]:
def evaluate_retrieval(bm25, test_queries, test_answers, corpus_answers,
                       k_values=[1, 3, 5, 10, 20]):
    """Evaluate BM25 retrieval: Recall@k and MRR@k."""
    results = {k: {'recall': [], 'mrr': []} for k in k_values}

    for query, gt_answer in zip(test_queries, test_answers):
        tokenized_query = tokenize(query)
        scores = bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1]

        for k in k_values:
            top_k = top_indices[:k]
            retrieved = [corpus_answers[i] for i in top_k]

            # Recall@k
            hit = gt_answer in retrieved
            results[k]['recall'].append(1.0 if hit else 0.0)

            # MRR@k
            rr = 0.0
            for rank, ans in enumerate(retrieved, 1):
                if ans == gt_answer:
                    rr = 1.0 / rank
                    break
            results[k]['mrr'].append(rr)

    return {k: {
        'recall': np.mean(v['recall']),
        'mrr': np.mean(v['mrr'])
    } for k, v in results.items()}


print(f"Evaluating {len(test_queries):,} queries ...")
t0 = time.time()
K_VALUES = [1, 3, 5, 10, 20]
eval_results = evaluate_retrieval(bm25, test_queries, test_answers,
                                   corpus_answers, K_VALUES)
eval_time = time.time() - t0

print(f"Evaluation done: {eval_time:.1f}s ({eval_time/len(test_queries)*1000:.1f} ms/query)")
print(f"\n{'k':>4}  {'Recall@k':>10}  {'MRR@k':>10}")
print('-' * 30)
for k in K_VALUES:
    r = eval_results[k]
    print(f"{k:>4}  {r['recall']:>10.4f}  {r['mrr']:>10.4f}")

In [ ]:
# --- Per-domain evaluation ---
domains = sorted(set(test_domains))
per_domain_results = {}

for domain in domains:
    idx = [i for i, d in enumerate(test_domains) if d == domain]
    dq = [test_queries[i] for i in idx]
    da = [test_answers[i] for i in idx]
    dr = evaluate_retrieval(bm25, dq, da, corpus_answers, K_VALUES)
    per_domain_results[domain] = {
        'n_queries': len(idx),
        'metrics': {k: {'recall': round(v['recall'], 4), 'mrr': round(v['mrr'], 4)}
                    for k, v in dr.items()}
    }

print("=== Per-Domain Recall@5 / MRR@5 ===")
for domain, info in per_domain_results.items():
    m = info['metrics'][5]
    print(f"  {domain:<16} n={info['n_queries']:>4}  "
          f"Recall@5={m['recall']:.4f}  MRR@5={m['mrr']:.4f}")

---
## 5. Results Visualization

In [ ]:
# --- Line chart: Recall@k and MRR@k vs k ---
recall_vals = [eval_results[k]['recall'] for k in K_VALUES]
mrr_vals = [eval_results[k]['mrr'] for k in K_VALUES]

fig = make_subplots(rows=1, cols=1)

fig.add_trace(go.Scatter(
    x=K_VALUES, y=recall_vals, mode='lines+markers+text',
    name='Recall@k',
    marker=dict(size=10, color='#42a5f5'),
    line=dict(width=3, color='#42a5f5'),
    text=[f'{v:.3f}' for v in recall_vals],
    textposition='top center',
))

fig.add_trace(go.Scatter(
    x=K_VALUES, y=mrr_vals, mode='lines+markers+text',
    name='MRR@k',
    marker=dict(size=10, color='#ffa726'),
    line=dict(width=3, color='#ffa726'),
    text=[f'{v:.3f}' for v in mrr_vals],
    textposition='bottom center',
))

fig.update_layout(
    title='BM25 Retrieval: Recall@k and MRR@k',
    xaxis_title='k (top-k)', yaxis_title='Score',
    xaxis=dict(tickvals=K_VALUES),
    yaxis=dict(range=[0, 1]),
    width=800, height=450,
    legend=dict(x=0.7, y=0.2),
)
fig.show(renderer='iframe')

In [ ]:
# --- Bar chart: Key metrics summary ---
summary_metrics = {
    'Recall@1': eval_results[1]['recall'],
    'Recall@5': eval_results[5]['recall'],
    'Recall@10': eval_results[10]['recall'],
    'Recall@20': eval_results[20]['recall'],
    'MRR@5': eval_results[5]['mrr'],
    'MRR@10': eval_results[10]['mrr'],
}

fig = go.Figure(go.Bar(
    x=list(summary_metrics.keys()),
    y=list(summary_metrics.values()),
    marker_color=['#42a5f5', '#42a5f5', '#42a5f5', '#42a5f5', '#ffa726', '#ffa726'],
    text=[f'{v:.4f}' for v in summary_metrics.values()],
    textposition='outside',
))
fig.update_layout(
    title='BM25 Retrieval: Key Metrics Summary',
    yaxis_title='Score', yaxis_range=[0, 1],
    width=750, height=420,
    margin=dict(t=60),
)
fig.show(renderer='iframe')

In [ ]:
# --- Per-domain retrieval performance (grouped bar) ---
domain_names = list(per_domain_results.keys())
domain_recall5 = [per_domain_results[d]['metrics'][5]['recall'] for d in domain_names]
domain_mrr5 = [per_domain_results[d]['metrics'][5]['mrr'] for d in domain_names]
domain_recall10 = [per_domain_results[d]['metrics'][10]['recall'] for d in domain_names]

fig = go.Figure()
fig.add_trace(go.Bar(name='Recall@5', x=domain_names, y=domain_recall5,
                     marker_color='#42a5f5',
                     text=[f'{v:.3f}' for v in domain_recall5], textposition='outside'))
fig.add_trace(go.Bar(name='Recall@10', x=domain_names, y=domain_recall10,
                     marker_color='#66bb6a',
                     text=[f'{v:.3f}' for v in domain_recall10], textposition='outside'))
fig.add_trace(go.Bar(name='MRR@5', x=domain_names, y=domain_mrr5,
                     marker_color='#ffa726',
                     text=[f'{v:.3f}' for v in domain_mrr5], textposition='outside'))

fig.update_layout(
    title='BM25 Per-Domain Retrieval Performance',
    xaxis_title='Domain', yaxis_title='Score',
    yaxis_range=[0, 1.05],
    barmode='group',
    width=900, height=480,
    xaxis_tickangle=-30,
    legend=dict(x=0.75, y=0.95),
)
fig.show(renderer='iframe')

In [ ]:
# --- Qualitative examples: top-5 retrieval results ---
print("=" * 80)
print("Sample Queries: Top-5 BM25 Retrieval Results")
print("=" * 80)

n_samples = min(5, len(test_queries))
sample_idx = np.random.choice(len(test_queries), size=n_samples, replace=False)

for si in sample_idx:
    query = test_queries[si]
    gt_answer = test_answers[si]
    tokenized_q = tokenize(query)
    scores = bm25.get_scores(tokenized_q)
    top5 = np.argsort(scores)[::-1][:5]

    print(f"\nQuery   : {query}")
    print(f"GT Answer: {gt_answer[:80]}..." if len(gt_answer) > 80 else f"GT Answer: {gt_answer}")
    print(f"Domain  : {test_domains[si]}")

    for rank, idx in enumerate(top5, 1):
        ans = corpus_answers[idx]
        score = scores[idx]
        match = ' [HIT]' if ans == gt_answer else ''
        ans_display = ans[:70] + '...' if len(ans) > 70 else ans
        print(f"  #{rank} (score={score:.2f}) {ans_display}{match}")
    print('-' * 80)

---
## 6. Save Results + Base64 Download

In [ ]:
# --- Build results JSON ---
bm25_results = {
    'method': 'BM25',
    'tokenizer': 'Mecab' if USE_MECAB else 'whitespace',
    'corpus_size': len(df_corpus),
    'test_size': len(test_queries),
    'eval_time_s': round(eval_time, 2),
    'metrics': {
        f'recall_at_{k}': round(eval_results[k]['recall'], 4)
        for k in K_VALUES
    },
    'per_domain': per_domain_results,
}

# Add MRR metrics
for k in K_VALUES:
    bm25_results['metrics'][f'mrr_at_{k}'] = round(eval_results[k]['mrr'], 4)

# Top-level convenience keys for backend consumption
bm25_results['recall_at_5'] = bm25_results['metrics']['recall_at_5']
bm25_results['mrr'] = bm25_results['metrics']['mrr_at_10']

# Save
results_path = os.path.join(RESULTS_DIR, 'retrieval_bm25_results.json')
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(bm25_results, f, ensure_ascii=False, indent=2)

print(f"Results saved: {results_path}")
print(f"\n=== Saved Metrics ===")
for k, v in bm25_results['metrics'].items():
    print(f"  {k}: {v}")

In [ ]:
# --- Base64 download helper ---
import base64
from IPython.display import display, HTML

def create_download_link(filepath, filename=None):
    """Create a clickable download link for any file."""
    if filename is None:
        filename = filepath.split('/')[-1]
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    size_mb = len(data) / 1024 / 1024
    href = (f'<a href="data:application/octet-stream;base64,{b64}" '
            f'download="{filename}">'
            f'\U0001f4e5 {filename} ({size_mb:.1f} MB)</a>')
    display(HTML(href))

create_download_link(results_path)

---
## 7. Summary

In [ ]:
print("=" * 60)
print("     06. BM25 Retrieval Baseline -- Summary")
print("=" * 60)

print(f"\nCorpus: {len(df_corpus):,} documents")
print(f"Test  : {len(test_queries):,} queries")
print(f"Tokenizer: {'Mecab' if USE_MECAB else 'Whitespace split'}")

print(f"\n{'Metric':<16} {'Value':>8}")
print('-' * 26)
for k, v in bm25_results['metrics'].items():
    print(f"{k:<16} {v:>8.4f}")

print(f"\n[Key Takeaways]")
print(f"  - BM25 is a keyword-matching baseline (TF-IDF family)")
print(f"  - Recall@5 = {bm25_results['metrics']['recall_at_5']:.4f}, "
      f"MRR@10 = {bm25_results['metrics']['mrr_at_10']:.4f}")
print(f"  - Semantic search (SBERT) should improve over keyword matching")
print(f"  - Particularly for paraphrase / intent-level queries")

print(f"\nArtifacts: {results_path}")
print(f"\nNext -> 07_sbert_faiss")
print("=" * 60)

---
## 8. BM25 선택 근거 및 설정 분석

### BM25Okapi 선택 이유

| Factor | Detail |
|--------|--------|
| **알고리즘** | BM25Okapi는 TF-IDF의 확률적 확장으로, 문서 길이 정규화(`dl/avgdl`)를 포함하여 긴 답변에 불이익을 주지 않음 |
| **파라미터** | `k1=1.5, b=0.75` (기본값). k1은 TF saturation 속도, b는 문서 길이 정규화 강도를 제어 |
| **토크나이저** | Mecab 형태소 분석기 우선 → 미설치 시 whitespace fallback. 한국어는 교착어이므로 형태소 분석이 정확한 매칭에 필수적 |
| **Baseline 역할** | 의미 검색(SBERT) 전에 키워드 매칭의 상한선을 확인. BM25 성능이 높으면 semantic gap이 작다는 의미 |

### 평가 설정

| Setting | Value | Rationale |
|---------|-------|-----------|
| `k_values` | [1, 3, 5, 10, 20] | k=1은 정밀도(Precision-oriented), k=20은 재현율(Recall-oriented) 평가. RAG에서는 top-3~5를 사용하므로 해당 구간이 핵심 |
| `N_TEST` | 1,000 | 전체 QA의 약 20%. 통계적으로 유의미한 평가를 위한 최소 샘플 수 |
| Corpus | 전체 QA (test 포함) | 실제 운영 환경에서는 전체 DB에서 검색하므로, test 쿼리가 corpus에 포함된 현실적 설정 |

---
## 9. Error Analysis — BM25 한계 및 SBERT 기대 효과

### BM25의 구조적 한계

1. **어휘 불일치 (Vocabulary Mismatch)**
   - "쓰레기 무단 투기" → "폐기물 불법 배출": 의미는 같지만 키워드가 다르면 검색 실패
   - 동의어/유의어 처리 불가 → Recall@1이 낮은 주요 원인

2. **문맥 무시 (Context Blindness)**
   - "도로 공사 소음" vs "도로 공사 진행": "도로 공사"는 동일하지만 의도가 다름
   - BM25는 단어 빈도만 보므로 문맥적 차이를 구분하지 못함

3. **도메인 편향 (Domain Bias)**
   - 고빈도 도메인의 문서가 더 많은 키워드 다양성을 가짐 → 해당 도메인의 Recall이 상대적으로 높음
   - 소수 도메인은 코퍼스 자체가 작아 BM25 점수 분포가 불리

### 07_sbert_faiss 기대 효과

| 개선 포인트 | 메커니즘 |
|------------|---------|
| **동의어 해결** | SBERT는 "쓰레기 투기" ≈ "폐기물 배출"을 유사 벡터로 매핑 |
| **의미 유사도** | 문장 전체의 의미를 768차원 벡터로 압축 → 키워드 무관 매칭 가능 |
| **도메인 균등화** | Dense 벡터는 키워드 빈도에 의존하지 않으므로 소수 도메인에서도 공정한 검색 |
| **속도** | FAISS IndexFlatIP + GPU → 수만 건 검색이 밀리초 단위 |

---
## 10. 데이터 편향 분석 — K쇼핑 과대 대표 문제

### 현황

| Domain | QA Pairs | 비율 |
|--------|----------|------|
| K쇼핑 | ~215,996 | **47.4%** |
| 질병관리본부 | ~100,773 | 22.1% |
| 금융/보험 | ~91,548 | 20.1% |
| 다산콜센터 | ~47,700 | 10.5% |

전체 QA 코퍼스의 **47%를 K쇼핑이 차지**합니다. 이는 BM25 검색 결과에 다음과 같은 편향을 유발할 수 있습니다:

### 편향 영향

1. **검색 결과 편향**: 도메인 무관 키워드("환불", "배송")가 K쇼핑 문서에 과도하게 매칭되어, 다른 도메인의 유사 질문에도 K쇼핑 답변이 상위에 랭킹될 가능성
2. **BM25 IDF 왜곡**: K쇼핑에서 자주 등장하는 용어의 IDF가 낮아져, 해당 용어가 다른 도메인에서는 중요한 키워드임에도 가중치가 감소
3. **Per-Domain 성능 격차**: 위 per-domain 평가에서 K쇼핑의 Recall이 상대적으로 높은 것은 코퍼스 내 후보 문서가 풍부하기 때문

### 완화 전략 (본 프로젝트에서 적용)

- **Classification → Retrieval 파이프라인**: 05번 노트북의 도메인 분류기가 먼저 도메인을 예측하고, 해당 도메인 내 QA에서만 검색 → K쇼핑 편향 차단
- **class_weight='balanced'**: 분류 단계에서 소수 도메인에 높은 가중치를 부여하여 분류 정확도 보장
- **07번 SBERT**: Dense 벡터 기반 검색은 키워드 빈도에 덜 의존하므로 IDF 왜곡 문제 완화

In [ ]:
# === Kaggle Dataset 자동 업로드 ===
if IS_KAGGLE:
    import shutil
    UPLOAD_DIR = '/kaggle/working/dataset_upload'
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    # 결과 파일 심볼릭 링크
    for src in [results_path]:
        dst = os.path.join(UPLOAD_DIR, os.path.basename(src))
        if os.path.exists(dst):
            os.remove(dst)
        os.symlink(src, dst)

    # 메타데이터
    meta = {
        "title": "civilcomplaint-bm25-retrieval",
        "id": "kukass/civilcomplaint-bm25-retrieval",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(os.path.join(UPLOAD_DIR, 'dataset-metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    !kaggle datasets create -p {UPLOAD_DIR} --dir-mode zip
    print("✅ Kaggle 데이터셋 업로드 완료: civilcomplaint-bm25-retrieval")
else:
    print("ℹ️ 로컬 환경 — Kaggle 업로드 건너뜀")